# 🎙️ Transkripsi Audio → Teks (Whisper)

Ubah rekaman **wawancara / ceramah / podcast** menjadi teks otomatis dengan
[faster-whisper](https://github.com/SYSTRAN/faster-whisper) — versi Whisper yang
4× lebih cepat, berjalan di GPU L40S. Mendukung **bahasa Indonesia**.

**Yang kamu pelajari:** memuat model dari folder bersama server (tanpa download),
transkripsi dengan *timestamp*, dan menyimpan hasil ke penyimpanan pribadimu.

In [ ]:
# Sel 1 — muat model. Server menyediakan model BERSAMA di /opt/ch-models
# (tidak perlu download!). Bila belum tersedia, otomatis download ukuran kecil.
import os
from faster_whisper import WhisperModel

shared = "/opt/ch-models/faster-whisper-small"
sumber = shared if os.path.isdir(shared) else "small"
model = WhisperModel(sumber, device="cuda", compute_type="float16")
print("Model siap dari:", sumber)

In [ ]:
# Sel 2 — siapkan audio. GANTI dengan file audiomu: klik ikon upload di
# explorer / seret file .mp3/.wav ke sini, lalu isi nama filenya di bawah.
# (Untuk demo tanpa file, sel ini membuat audio ucapan sintetis sederhana.)
AUDIO = "audio_saya.mp3"   # <-- ganti dengan nama file audiomu

import numpy as np, soundfile as sf
if not os.path.exists(AUDIO):
    sr = 16000
    t = np.linspace(0, 2.0, 2 * sr)
    gelombang = 0.05 * np.sin(2 * np.pi * 220 * t)   # nada uji 2 detik
    sf.write("contoh_nada.wav", gelombang, sr)
    AUDIO = "contoh_nada.wav"
    print("File audiomu belum ada — memakai nada uji (hasil transkrip akan kosong).")
print("Audio:", AUDIO)

In [ ]:
# Sel 3 — transkripsi + timestamp per segmen.
segments, info = model.transcribe(AUDIO, language="id", vad_filter=True)
print(f"Terdeteksi bahasa: {info.language} (probabilitas {info.language_probability:.0%})\n")

baris = []
for seg in segments:
    baris.append(f"[{seg.start:6.1f}s → {seg.end:6.1f}s] {seg.text.strip()}")
    print(baris[-1])
if not baris:
    print("(tidak ada ucapan terdeteksi — unggah rekaman asli di Sel 2)")

In [ ]:
# Sel 4 — simpan transkrip ke penyimpanan pribadimu (menu Penyimpanan).
from pathlib import Path
out = Path("/persist/transkrip.txt")
out.write_text("\n".join(baris) or "(kosong)", encoding="utf-8")
print("Tersimpan:", out)

**Langkah lanjut:** untuk akurasi maksimal ganti ke model besar:
`WhisperModel("/opt/ch-models/faster-whisper-large-v3", device="cuda", compute_type="float16")`
— tetap tanpa download karena sudah disediakan server. Ukur akurasi dengan
`jiwer` (Word Error Rate) bila kamu punya transkrip acuan.